In [2]:
import pandas as pd
from seaborn.external.docscrape import header
!pip install kagglehub opencv-python seaborn matplotlib scikit-learn

  Using cached opencv_python-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached kagglesdk-0.1.15-py3-none-any.whl.metadata (13 kB)
  Using cached tqdm-4.67.2-py3-none-any.whl.metadata (57 kB)
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinu

In [3]:
import kagglehub
import os

# Define the target directory
download_dir = './dataset'
os.makedirs(download_dir, exist_ok=True)

download_path = kagglehub.dataset_download(
    'mdraselsarker/mot15-challenge-dataset',
)

print(f"Dataset downloaded to: {download_path}")


/home/yogendra/workspace/python/university/Object-detection-and-tracking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.22G/1.22G [05:45<00:00, 3.79MB/s]

Extracting files...


Dataset downloaded to: /home/yogendra/.cache/kagglehub/datasets/mdraselsarker/mot15-challenge-dataset/versions/1


In [1]:
import os
import pandas as pd
# name=ADL-Rundle-6
# imDir=img1
# frameRate=30
# seqLength=525
# imWidth=1920
# imHeight=1080
# imExt=.jpg

base_dir = './dataset/'
video_test_df = pd.DataFrame(columns=['clip_name', 'clip_path', 'frameRate', 'seqLength', 'imWidth', 'imHeight', 'imExt'])
# train_df = pd.DataFrame(columns=['frame_id', 'id', 'x', 'y', 'w', 'h', 'conf', 'cls_id'])

In [20]:
object_detection_columns = [
            "frame_id",
            "object_id",
            "x_coordinate",
            "y_coordinate",
            "width",
            "height",
            "confidence",
            "x_init",
            "y_init",
            "z_init"
        ]

In [42]:
import configparser

config = configparser.ConfigParser()
def process_train_df(folder_path: str):
    folders = sorted(os.listdir(folder_path))
    features = []
    object_detection_df = pd.DataFrame(columns=object_detection_columns)
    ground_truth_detection_df = pd.DataFrame(columns=object_detection_columns)

    for folder_id, folder in enumerate(folders):
        print(f"File - {folder}")

        config_file_path = folder_path + folder + "/seqinfo.ini"
        config.read(config_file_path)

        metadata = dict(config.items('Sequence'))
        for key in ["name", "imdir", "framerate", "seqlength", "imwidth", "imheight", "imext"]:
            if key not in metadata:
                raise Exception(f"Key {key} not found in seqinfo.ini in folder {folder}")

        top_level_feat = {
                "video_id": folder_id,
                "video_name": metadata["name"],
                "image_width": metadata["imwidth"],
                "image_height": metadata["imheight"],
            }

        image_dir_path = folder_path + folder + "/" + metadata["imdir"]
        image_files = sorted(os.listdir(image_dir_path))
        for frame_id, image_file in enumerate(image_files):
            features.append({
                **top_level_feat,
                "frame_id": frame_id,
                "image_path": image_dir_path + "/" + image_file
            })

        detection_file_path = folder_path + folder + "/det/det.txt"
        det_df = pd.read_csv(detection_file_path, header=None, names = object_detection_columns)
        det_df["video_id"] = folder_id
        object_detection_df = pd.concat([object_detection_df, det_df], axis=0)

        gt_file_path = folder_path + folder + "/gt/gt.txt"
        gt_df = pd.read_csv(gt_file_path, header=None, names = object_detection_columns)
        gt_df["video_id"] = folder_id
        ground_truth_detection_df = pd.concat([ground_truth_detection_df, gt_df], axis=0)

    video_train_df = pd.DataFrame(columns=['video_id', 'video_name', 'image_width', 'image_height', 'frame_id', 'image_path'], data=features)

    object_detection_df['video_id'] = object_detection_df['video_id'].astype(int)
    ground_truth_detection_df['video_id'] = ground_truth_detection_df['video_id'].astype(int)
    object_detection_df['frame_id'] = object_detection_df['frame_id'].map(lambda x: x - 1)
    ground_truth_detection_df['frame_id'] = ground_truth_detection_df['frame_id'].map(lambda x: x - 1)
    return video_train_df, object_detection_df, ground_truth_detection_df

train_df, det_df, gt_df = process_train_df(base_dir + 'train/')
train_df.to_csv('./dataset/train.csv')
det_df.to_csv('./dataset/train_det.csv')
gt_df.to_csv('./dataset/train_gt.csv')

File - ADL-Rundle-6
File - ADL-Rundle-8
File - ETH-Bahnhof
File - ETH-Pedcross2
File - ETH-Sunnyday
File - KITTI-13
File - KITTI-17
File - PETS09-S2L1
File - TUD-Campus
File - TUD-Stadtmitte
File - Venice-2


In [39]:
det_df.head(1000)

,frame_id,object_id,x_coordinate,y_coordinate,width,height,confidence,x_init,y_init,z_init,video_id
0,0,-1,1689,385,146.62,332.71,67.567,-1,-1,-1,0
1,0,-1,1303,503,61.514,139.59,29.439,-1,-1,-1,0
2,0,-1,1258,569,40.123,91.049,19.601,-1,-1,-1,0
3,0,-1,31,525,113.37,257.27,17.013,-1,-1,-1,0
4,0,-1,1800,483,94.66,214.81,11.949,-1,-1,-1,0
...,...,...,...,...,...,...,...,...,...,...,...
995,134,-1,1142,391,250.0,567.31,28.757,-1,-1,-1,0
996,134,-1,318,438,123.42,280.06,27.072,-1,-1,-1,0
997,134,-1,1367,171,270.83,614.58,25.287,-1,-1,-1,0
998,134,-1,1395,610,146.62,332.71,21.756,-1,-1,-1,0


In [12]:
df.groupby('video_id').count()

,video_name,image_width,image_height,frame_id,image_path
video_id,,,,,
0,525,525,525,525,525
1,654,654,654,654,654
2,1000,1000,1000,1000,1000
3,837,837,837,837,837
4,354,354,354,354,354
5,340,340,340,340,340
6,145,145,145,145,145
7,795,795,795,795,795
8,71,71,71,71,71


In [15]:
detection_file_path = "./dataset/train/ADL-Rundle-6" + "/det/det.txt"
detection_df = pd.read_csv(detection_file_path, header=None, names = [
            "frame_id",
            "object_id",
            "x_coordinate",
            "y_coordinate",
            "width",
            "height",
            "confidence",
            "x_init",
            "y_init",
            "z_init"
        ])
